In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')


Libraries imported successfully!


In [2]:
count_points = pd.read_csv('../count_points.csv', low_memory=False)
print('Count Points Shape:', count_points.shape)
print('\nColumns:', count_points.columns.tolist())
count_points.head()

Count Points Shape: (46251, 19)

Columns: ['count_point_id', 'year', 'region_id', 'region_name', 'region_ons_code', 'local_authority_id', 'local_authority_name', 'local_authority_code', 'road_name', 'road_category', 'road_type', 'start_junction_road_name', 'end_junction_road_name', 'easting', 'northing', 'latitude', 'longitude', 'link_length_km', 'link_length_miles']


,count_point_id,year,region_id,region_name,region_ons_code,local_authority_id,local_authority_name,local_authority_code,road_name,road_category,road_type,start_junction_road_name,end_junction_road_name,easting,northing,latitude,longitude,link_length_km,link_length_miles
0,51,2024,1,South West,E12000009,1,Isles of Scilly,E06000053,A3111,PA,Major,"Pierhead, Hugh Town",A3112,90173,10641,49.915503,-6.317558,0.3,0.19
1,52,2024,1,South West,E12000009,1,Isles of Scilly,E06000053,A3112,PA,Major,A3111,A3110,91203,10217,49.912233,-6.302913,2.0,1.24
2,53,2024,1,South West,E12000009,1,Isles of Scilly,E06000053,A3111,PA,Major,A3112,A3110,90782,10687,49.916231,-6.309136,1.2,0.75
3,54,2024,1,South West,E12000009,1,Isles of Scilly,E06000053,A3110,PA,Major,A3111,A3112,91515,10820,49.917802,-6.299062,0.3,0.19
4,55,2024,1,South West,E12000009,1,Isles of Scilly,E06000053,A3110,PA,Major,A3111,A3112,91664,10833,49.917996,-6.297002,4.0,2.49


In [3]:
raw_counts = pd.read_csv('../dft_traffic_counts_raw_counts.csv', low_memory=False)
print('Raw Counts Shape:', raw_counts.shape)
print('\nColumns:', raw_counts.columns.tolist())
raw_counts.head()

Raw Counts Shape: (5113740, 35)

Columns: ['count_point_id', 'direction_of_travel', 'year', 'count_date', 'hour', 'region_id', 'region_name', 'region_ons_code', 'local_authority_id', 'local_authority_name', 'local_authority_code', 'road_name', 'road_category', 'road_type', 'start_junction_road_name', 'end_junction_road_name', 'easting', 'northing', 'latitude', 'longitude', 'link_length_km', 'link_length_miles', 'pedal_cycles', 'two_wheeled_motor_vehicles', 'cars_and_taxis', 'buses_and_coaches', 'LGVs', 'HGVs_2_rigid_axle', 'HGVs_3_rigid_axle', 'HGVs_4_or_more_rigid_axle', 'HGVs_3_or_4_articulated_axle', 'HGVs_5_articulated_axle', 'HGVs_6_articulated_axle', 'all_HGVs', 'all_motor_vehicles']


,count_point_id,direction_of_travel,year,count_date,hour,region_id,region_name,region_ons_code,local_authority_id,local_authority_name,...,buses_and_coaches,LGVs,HGVs_2_rigid_axle,HGVs_3_rigid_axle,HGVs_4_or_more_rigid_axle,HGVs_3_or_4_articulated_axle,HGVs_5_articulated_axle,HGVs_6_articulated_axle,all_HGVs,all_motor_vehicles
0,51,S,2004,2004-05-21,11,1,South West,E12000009,1,Isles of Scilly,...,2.0,16,2.0,0.0,0.0,0.0,0,0.0,2.0,49.0
1,51,S,2004,2004-05-21,15,1,South West,E12000009,1,Isles of Scilly,...,1.0,13,2.0,0.0,0.0,0.0,0,0.0,2.0,46.0
2,51,S,2004,2004-05-21,13,1,South West,E12000009,1,Isles of Scilly,...,2.0,23,5.0,0.0,0.0,0.0,0,0.0,5.0,51.0
3,51,N,2004,2004-05-21,7,1,South West,E12000009,1,Isles of Scilly,...,1.0,13,0.0,0.0,0.0,0.0,0,0.0,0.0,19.0
4,51,N,2004,2004-05-21,10,1,South West,E12000009,1,Isles of Scilly,...,4.0,4,4.0,0.0,0.0,0.0,0,0.0,4.0,41.0


In [4]:
london_cp = count_points[
    count_points['region_name'].str.contains('London', na=False)
]
london_ids = london_cp['count_point_id'].unique().tolist()

print(f'Total London count points: {len(london_ids)}')
print(f'\nLondon boroughs covered ({len(london_cp["local_authority_name"].unique())} boroughs):')
print(london_cp['local_authority_name'].unique())

Total London count points: 3630

London boroughs covered (33 boroughs):
['Barnet' 'Hillingdon' 'Tower Hamlets' 'Islington' 'Southwark' 'Lewisham'
 'Greenwich' 'Bexley' 'Lambeth' 'Wandsworth' 'Westminster'
 'Kensington and Chelsea' 'Hounslow' 'Brent' 'Enfield' 'Hackney'
 'Redbridge' 'Waltham Forest' 'Croydon' 'Camden' 'Hammersmith and Fulham'
 'Ealing' 'Haringey' 'Newham' 'Barking and Dagenham' 'City of London'
 'Richmond upon Thames' 'Bromley' 'Sutton' 'Kingston upon Thames' 'Harrow'
 'Havering' 'Merton']


In [5]:
london_traffic = raw_counts[
    raw_counts['count_point_id'].isin(london_ids)
].copy()

print(f'London traffic records: {len(london_traffic):,}')
print(f'Unique junctions:       {london_traffic["count_point_id"].nunique():,}')
print(f'Year range:             {london_traffic["year"].min()} to {london_traffic["year"].max()}')
print(f'\nColumns:')
print(london_traffic.dtypes)

London traffic records: 414,252
Unique junctions:       3,359
Year range:             2000 to 2024

Columns:
count_point_id                    int64
direction_of_travel              object
year                              int64
count_date                       object
hour                              int64
region_id                         int64
region_name                      object
region_ons_code                  object
local_authority_id                int64
local_authority_name             object
local_authority_code             object
road_name                        object
road_category                    object
road_type                        object
start_junction_road_name         object
end_junction_road_name           object
easting                           int64
northing                          int64
latitude                        float64
longitude                       float64
link_length_km                  float64
link_length_miles               float64
pedal_cycle

In [6]:
df = london_traffic.rename(columns={
    'count_point_id':    'Junction',
    'all_motor_vehicles': 'Vehicles'
}).copy()

# Drop rows with missing vehicle counts
df = df.dropna(subset=['Vehicles'])

print(f'Final dataset shape: {df.shape}')
print(f'\nVehicle count statistics:')
print(df['Vehicles'].describe())

Final dataset shape: (414251, 35)

Vehicle count statistics:
count    414251.000000
mean        738.456264
std         902.942942
min           0.000000
25%         133.000000
50%         470.000000
75%         897.000000
max       10905.000000
Name: Vehicles, dtype: float64


In [7]:
time_cols = [c for c in df.columns if any(t in c.lower()
             for t in ['year', 'month', 'day', 'hour', 'date', 'time'])]
print('Time columns found:', time_cols)
print('\nSample values:')
print(df[time_cols].head())

Time columns found: ['year', 'count_date', 'hour']

Sample values:
       year  count_date  hour
30192  2000  2000-03-27     7
30193  2000  2000-03-27    14
30194  2000  2000-03-27    17
30195  2000  2000-03-27     9
30196  2000  2000-03-27    16


In [8]:
df['Date'] = pd.to_datetime(df['count_date'])

df['day']     = df['Date'].dt.day
df['month']   = df['Date'].dt.month
df['weekday'] = df['Date'].dt.weekday

print('DateTime created!')
print(f'Date range: {df["Date"].min()} → {df["Date"].max()}')
df[['latitude','longitude', 'hour', 'day', 'month', 'weekday', 'Vehicles']].head(10)

DateTime created!
Date range: 2000-03-17 00:00:00 → 2024-11-06 00:00:00


,latitude,longitude,hour,day,month,weekday,Vehicles
30192,51.587522,-0.237927,7,27,3,0,2311.0
30193,51.587522,-0.237927,14,27,3,0,1559.0
30194,51.587522,-0.237927,17,27,3,0,710.0
30195,51.587522,-0.237927,9,27,3,0,2270.0
30196,51.587522,-0.237927,16,27,3,0,936.0
30197,51.587522,-0.237927,14,27,3,0,737.0
30198,51.587522,-0.237927,15,27,3,0,528.0
30199,51.587522,-0.237927,9,27,3,0,945.0
30200,51.587522,-0.237927,8,27,3,0,1633.0
30201,51.587522,-0.237927,8,27,3,0,1469.0


In [10]:
print(f'Total records:      {len(df):,}')
print(f'Total junctions:    {df["Junction"].nunique():,}')
print(f'Year range:         {df["Date"].dt.year.min()} → {df["Date"].dt.year.max()}')
print(f'Avg vehicles/hr:    {df["Vehicles"].mean():.0f}')
print(f'Max vehicles/hr:    {df["Vehicles"].max():.0f}')

Total records:      414,251
Total junctions:    3,359
Year range:         2000 → 2024
Avg vehicles/hr:    738
Max vehicles/hr:    10905


In [11]:
results = []

def evaluate(name, Y_test, y_pred, Y_train, y_pred_train):
    test_mae  = mean_absolute_error(Y_test, y_pred)
    test_rmse = np.sqrt(mean_squared_error(Y_test, y_pred))
    test_r2   = r2_score(Y_test, y_pred)
    train_r2  = r2_score(Y_train, y_pred_train)
    gap       = abs(train_r2 - test_r2)

    print(f'\n{"-"*55}')
    print(f'  Model: {name}')
    print(f'{"-"*55}')
    print(f'  For Test .....')
    print(f'  MAE:  {test_mae:.4f}')
    print(f'  RMSE: {test_rmse:.4f}')
    print(f'  R2:   {test_r2:.4f}')
    print(f'  For Train .....')
    print(f'  MAE:  {mean_absolute_error(Y_train, y_pred_train):.4f}')
    print(f'  RMSE: {np.sqrt(mean_squared_error(Y_train, y_pred_train)):.4f}')
    print(f'  R2:   {train_r2:.4f}')
    print(f'  Gap:  {gap:.4f}')

    results.append({
        'Model':     name,
        'Test MAE':  round(test_mae, 4),
        'Test RMSE': round(test_rmse, 4),
        'Test R²':   round(test_r2, 4),
        'Train R²':  round(train_r2, 4),
        'Gap':       round(gap, 4)
    })
    return test_r2

print('Evaluate function ready!')

Evaluate function ready!


In [12]:
X_base = df[['latitude', 'longitude', 'hour', 'day', 'month', 'weekday']]
Y_base = df['Vehicles']

# shuffle=True — safe since no lag features
X_base_train, X_base_test, Y_base_train, Y_base_test = train_test_split(
    X_base, Y_base, test_size=0.2, random_state=42, shuffle=True
)


In [13]:


model = DecisionTreeRegressor(random_state=42)
model.fit(X_base_train, Y_base_train)

evaluate(
    'Decision Tree',
    Y_base_test, model.predict(X_base_test),
    Y_base_train, model.predict(X_base_train)
)


-------------------------------------------------------
  Model: Decision Tree
-------------------------------------------------------
  For Test .....
  MAE:  172.8294
  RMSE: 342.1004
  R2:   0.8583
  For Train .....
  MAE:  74.3630
  RMSE: 162.5087
  R2:   0.9675
  Gap:  0.1092


0.8583466147455039